# Machine Sound Fault Detection - Supervised CNN + Augmentation (MIMII)

**This notebook is different from the other two.** It uses **faulty clips for training**.
The project rule (`CLAUDE.md`) says "train on healthy sound only". This notebook breaks that rule **on purpose**, as a separate experiment.
Never mix its numbers with the healthy-only results.

**What it does**
1. For each machine and ID, it splits the clips (healthy **and** faulty) into train / validation / test.
2. A small CNN learns to say **healthy or faulty** from a picture of the sound.
3. It runs each case **twice**: without and with data augmentation, so we can see if augmentation helps.
4. It draws a full report **per machine** and a bigger report for **all machines together**.

**What you need to do**
1. Add the MIMII dataset (**Add Input**).
2. Turn on the GPU (**Settings > Accelerator > GPU**).
3. Click **Run All**.

**Default = full run:** 4 machines x 4 IDs (id_00, id_02, id_04, id_06) x 6 dB = 16 runs, each trained twice (without and with augmentation).
**Want a fast check first?** Set `QUICK = True` in Step 2 (2 IDs, 150 clips per class).

The default split is **blocked** (the strict one). `random` is easier and gives higher, less trustworthy scores.

## Words used in this notebook

| Word | Meaning |
|---|---|
| **clip** | One 10-second sound file |
| **window** | A 2-second piece of a clip. The CNN looks at windows |
| **score** | Chance of "faulty" for a clip (0 to 1) |
| **augmentation** | Small random changes to training sound (louder, shifted, a bit of noise) |
| **loss** | The training error. It should go down |
| **AUC** | How well the score separates healthy from faulty. 1.0 = perfect, 0.5 = guess |
| **balanced accuracy** | Average of "faults caught" and "healthy kept". Fair when classes are not equal |
| **recall** | Share of faulty clips we catch |
| **false alarm** | Share of healthy clips we wrongly call faulty |

## Step 1 - Import tools

In [ ]:
import glob, gc, re, sys, time, os, subprocess, importlib.util, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from joblib import Parallel, delayed
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, confusion_matrix, ConfusionMatrixDisplay)
from IPython.display import display

for name in ("librosa", "soundfile"):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
import librosa
import soundfile as sf

## Step 2 - Settings

- `QUICK = False` is the full run. `QUICK = True` is a fast check (2 IDs, 150 clips per class).
- `SPLIT_MODE = "blocked"` is the strict split (see Step 8). Use `"random"` only to compare.

In [ ]:
# ---- Where things are ----
DATA_ROOT = "/kaggle/input"
WORK_DIR  = "/kaggle/working"

# ---- What to run ----
MACHINES  = ["fan", "pump", "slider", "valve"]
QUICK     = False                    # True = fast check (2 IDs, 150 clips per class). False = full run
IDS       = ["id_00", "id_02"] if QUICK else ["id_00", "id_02", "id_04", "id_06"]
LIMIT     = 150 if QUICK else None   # max clips per class per run (random sample)
SNR       = 6                        # noise level: 6, 0 or -6
SPLIT_MODE = "blocked"               # "blocked" (strict, recommended) or "random" (see Step 8)
AUGMENT_RUNS = [False, True]         # train without and with augmentation
HEADLINE_AUG = True                  # which one the main charts show (decided before running)
MAX_HOURS = 10

# ---- Sound settings ----
SAMPLE_RATE = 16000
N_FFT, HOP, N_MELS = 1024, 512, 128
MIC     = 0
WINDOW  = 32768
STEP    = 16384
FRAMES  = 64

# ---- Augmentation (only on training data) ----
GAIN_DB  = 2.0                       # random volume change, +/- 2 dB (as in CLAUDE.md)
NOISE_DB = 1.0                       # light random noise on the picture
SHIFT    = 8                         # random shift in time, up to 8 columns

# ---- Training ----
BATCH = 128
EPOCHS = 40
LEARNING_RATE = 1e-3
SEED = 42

MACHINE_COLOR = {"fan": "#1f77b4", "pump": "#ff7f0e", "slider": "#2ca02c", "valve": "#d62728"}
FIG_DIR = WORK_DIR + "/figures"
os.makedirs(FIG_DIR, exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=110, bbox_inches="tight")
    plt.show()

## Step 3 - Check the GPU

In [ ]:
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("TensorFlow", tf.__version__)
print("GPUs found:", [g.name for g in gpus])

## Step 4 - Find the audio files

In [ ]:
def read_path(path):
    parts = path.split("/")[::-1]          # start from the file name
    label   = next((p for p in parts if p in ("normal", "abnormal")), None)
    machine = next((p for p in parts if p in ("fan", "pump", "slider", "valve")), None)
    mid     = next((p for p in parts if re.fullmatch(r"id_\d+", p)), None)
    snr = None
    for p in parts:
        m = re.match(r"(-?\d+)\s*_?dB", p)
        if m:
            snr = int(m.group(1))
            break
    return label, machine, mid, snr

rows = []
for path in glob.glob(DATA_ROOT + "/**/*.wav", recursive=True):
    label, machine, mid, snr = read_path(path)
    if label and machine and mid:
        rows.append([path, machine, mid, label, snr])

df = pd.DataFrame(rows, columns=["path", "machine", "id", "label", "snr"])
assert len(df) > 0, "No audio found. Did you add the dataset to this notebook (Add Input)?"
print(len(df), "clips found")

Number of clips per machine, ID, noise level and label:

In [ ]:
counts = df.groupby(["machine", "id", "snr", "label"], dropna=False).size().unstack("label", fill_value=0)
display(counts)

## Step 5 - Choose the runs

If `LIMIT` is set, we take a **random** sample of that many clips per class (same sample every time).

In [ ]:
USE_SNR = df["snr"].notna().any()

def get_files(machine, mid):
    rows = df[(df["machine"] == machine) & (df["id"] == mid)]
    if USE_SNR:
        rows = rows[rows["snr"] == SNR]
    healthy = sorted(rows[rows["label"] == "normal"]["path"])
    faulty  = sorted(rows[rows["label"] == "abnormal"]["path"])
    if LIMIT:
        rng = np.random.default_rng(SEED)
        if len(healthy) > LIMIT: healthy = sorted(rng.choice(healthy, LIMIT, replace=False).tolist())
        if len(faulty)  > LIMIT: faulty  = sorted(rng.choice(faulty,  LIMIT, replace=False).tolist())
    return healthy, faulty

plan = []
for machine in MACHINES:
    for mid in IDS:
        healthy, faulty = get_files(machine, mid)
        if len(healthy) >= 30 and len(faulty) >= 30:
            plan.append((machine, mid))
        else:
            print("skip", machine, mid, "- not enough clips")
print(len(plan), "runs planned (each runs", len(AUGMENT_RUNS), "times):")
print(plan)

## Step 6 - Turn sound into a picture

A **log-mel picture**: 128 rows (pitch) x 64 columns (time) for each 2-second window.

In [ ]:
def load_audio(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)   # shape: (samples, microphones)
    audio = audio[:, min(MIC, audio.shape[1] - 1)]                 # keep one microphone
    if rate != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=rate, target_sr=SAMPLE_RATE)
    return audio

def cut_windows(audio):
    if len(audio) < WINDOW:
        audio = np.pad(audio, (0, WINDOW - len(audio)))
    starts = range(0, len(audio) - WINDOW + 1, STEP)
    return [audio[s:s + WINDOW] for s in starts]

def to_logmel(window):
    mel = librosa.feature.melspectrogram(y=window, sr=SAMPLE_RATE, n_fft=N_FFT,
                                         hop_length=HOP, n_mels=N_MELS)
    db = librosa.power_to_db(mel, ref=1.0)
    return db[:, :FRAMES]                                          # shape: (128, 64)

Example: one healthy and one faulty window.

In [ ]:
machine, mid = plan[0]
healthy, faulty = get_files(machine, mid)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, path, name in zip(axs, [healthy[0], faulty[0]], ["healthy", "faulty"]):
    ax.imshow(to_logmel(load_audio(path)[:WINDOW]), origin="lower", aspect="auto", cmap="magma")
    ax.set(title=f"{machine} {mid} - {name}", xlabel="time", ylabel="mel band")
show("example_spectrograms")

## Step 7 - Load the clips

In [ ]:
def clip_to_pictures(path):
    windows = cut_windows(load_audio(path))
    return np.array([to_logmel(w) for w in windows], dtype=np.float32)

def load_clips(files):
    pieces = Parallel(n_jobs=-1)(delayed(clip_to_pictures)(f) for f in files)
    pictures = np.concatenate(pieces)                                          # all windows together
    clip_id = np.concatenate([np.full(len(p), i) for i, p in enumerate(pieces)])  # which clip each window came from
    return pictures, clip_id

## Step 8 - Split the clips

Each class (healthy, faulty) is split **by clip**: **60% train, 10% validation, 30% test**.
No clip is ever in two sets.

- `random`: clips are picked at random.
- `blocked`: clips are cut in file-name order (first 60% train, next 10% validation, last 30% test). My guess is that file order follows recording time, so this is a **stricter** test (neighbours in time do not share sets). I have not checked this guess.

If `random` gives a much better score than `blocked`, the model may be using recording conditions, not the fault.

In [ ]:
def split_class(files):
    files = sorted(files)
    n = len(files)
    order = np.random.default_rng(SEED).permutation(n) if SPLIT_MODE == "random" else np.arange(n)
    n_train, n_val = int(0.6 * n), int(0.1 * n)
    pick = lambda idx: [files[i] for i in idx]
    return pick(order[:n_train]), pick(order[n_train:n_train + n_val]), pick(order[n_train + n_val:])

## Step 9 - Augmentation and the model input

**Augmentation** (training only, never on validation or test):
- random volume change (+/- 2 dB),
- a light random noise on the picture,
- a small random shift in time.

None of these can make a healthy sound look faulty.

In [ ]:
def make_prepare(mean, std, augment):
    mean_t, std_t = tf.constant(mean), tf.constant(std)
    def prepare(x, y):
        if augment:
            b = tf.shape(x)[0]
            x = x + tf.random.uniform((b, 1, 1), -GAIN_DB, GAIN_DB)               # volume
            x = x + tf.random.normal(tf.shape(x), stddev=NOISE_DB)                # noise
            x = tf.roll(x, tf.random.uniform((), -SHIFT, SHIFT + 1, dtype=tf.int32), axis=2)  # shift
        x = (x - mean_t) / std_t
        return x[..., None], y
    return prepare

def make_dataset(pictures, labels, mean, std, augment, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((pictures, labels.astype(np.float32)))
    if shuffle:
        ds = ds.shuffle(len(pictures), seed=SEED)
    ds = ds.batch(BATCH).map(make_prepare(mean, std, augment))
    return ds.prefetch(tf.data.AUTOTUNE)

## Step 10 - The model

A small CNN: 3 blocks of convolution, then one number between 0 and 1 (chance of faulty).

In [ ]:
def build_cnn():
    inputs = tf.keras.Input(shape=(N_MELS, FRAMES, 1))
    x = inputs
    for filters in (16, 32, 64):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.MaxPool2D(2)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs, outputs)

build_cnn().summary()

## Step 11 - Train

- Early stopping and learning-rate reduction watch the **validation loss**.
- Faulty clips are fewer in some runs, so each class gets a **weight** to balance them.

In [ ]:
def train_model(X_train, y_train, X_val, y_val, mean, std, augment):
    model = build_cnn()
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="binary_crossentropy", metrics=["accuracy"])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    ]
    n = len(y_train)
    weights = {0: n / (2 * (y_train == 0).sum()), 1: n / (2 * (y_train == 1).sum())}
    history = model.fit(make_dataset(X_train, y_train, mean, std, augment, shuffle=True),
                        validation_data=make_dataset(X_val, y_val, mean, std, False, shuffle=False),
                        epochs=EPOCHS, callbacks=callbacks, class_weight=weights, verbose=0)
    return model, history.history

## Step 12 - Score the clips

Each window gets a chance of "faulty". A clip score is the **average** of its window scores.

In [ ]:
def clip_scores(model, pictures, clip_id, mean, std):
    x = ((pictures - mean) / std)[..., None]
    window_scores = model.predict(x, batch_size=512, verbose=0).ravel()
    return pd.Series(window_scores).groupby(clip_id).mean().values

## Step 13 - Measure the result

**Threshold:** the score that gives the best **balanced accuracy on the validation set**.
Validation holds faulty clips here, so we can pick it. The test set is never used to pick it.

In [ ]:
def pick_threshold(val_scores, y_val):
    grid = np.linspace(0.01, 0.99, 99)
    bal = []
    for t in grid:
        pred = val_scores > t
        bal.append(0.5 * (pred[y_val == 1].mean() + (~pred)[y_val == 0].mean()))
    bal = np.array(bal)
    best = grid[bal == bal.max()]
    return float(np.median(best))

def get_metrics(val_scores, y_val, test_scores, y_test):
    threshold = pick_threshold(val_scores, y_val)
    pred = (test_scores > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y_test, test_scores)
    return {
        "auc":  roc_auc_score(y_test, test_scores),
        "pauc": roc_auc_score(y_test, test_scores, max_fpr=0.1),
        "f1":   f1_score(y_test, pred),
        "accuracy": (tp + tn) / len(y_test),
        "bal_acc": 0.5 * (tp / (tp + fn) + tn / (tn + fp)),
        "recall": tp / (tp + fn),
        "false_alarm": fp / (fp + tn),
        "recall_at_5pct_fa": float(np.interp(0.05, fpr, tpr)),
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

In [ ]:
# Published AUC from the MIMII paper (Table 3), for comparison.
# Order inside each tuple: 6 dB, 0 dB, -6 dB
PUBLISHED = {
 "fan":    {"id_00": (.75,.63,.57), "id_01": (.97,.90,.70), "id_02": (.99,.83,.68), "id_03": (1.0,.89,.70),
            "id_04": (.92,.75,.57), "id_05": (.95,.90,.83), "id_06": (.99,.97,.83)},
 "pump":   {"id_00": (.84,.65,.58), "id_01": (.98,.90,.73), "id_02": (.45,.46,.52), "id_03": (.79,.81,.75),
            "id_04": (.99,.95,.93), "id_05": (.66,.66,.64), "id_06": (.94,.76,.61)},
 "slider": {"id_00": (.99,.99,.93), "id_01": (.94,.90,.83), "id_02": (.93,.79,.74), "id_03": (.99,.85,.71),
            "id_04": (.88,.78,.61), "id_05": (.84,.70,.60), "id_06": (.71,.56,.52)},
 "valve":  {"id_00": (.68,.55,.62), "id_01": (.77,.71,.61), "id_02": (.66,.59,.57), "id_03": (.70,.65,.44),
            "id_04": (.64,.65,.50), "id_05": (.52,.48,.44), "id_06": (.70,.66,.53)},
}
SNR_COLUMN = {6: 0, 0: 1, -6: 2}

## Step 14 - One full run

For one machine + one ID: load, split, then train and measure **once per augmentation setting**.
Pictures are made only once and used for both.

In [ ]:
def run_one(machine, mid):
    healthy, faulty = get_files(machine, mid)
    h_tr, h_va, h_te = split_class(healthy)
    f_tr, f_va, f_te = split_class(faulty)
    print(f"  healthy train/val/test {len(h_tr)}/{len(h_va)}/{len(h_te)} | faulty {len(f_tr)}/{len(f_va)}/{len(f_te)}")

    def load(h_files, f_files):
        pics, cid = load_clips(h_files + f_files)
        y_clip = np.array([0] * len(h_files) + [1] * len(f_files))
        return pics, cid, y_clip, y_clip[cid]                  # last one: label of each window
    X_tr, _,    y_tr_clip, y_tr = load(h_tr, f_tr)
    X_va, c_va, y_va,      y_va_w = load(h_va, f_va)
    X_te, c_te, y_te,      y_te_w = load(h_te, f_te)

    mean = X_tr.mean(axis=(0, 2), keepdims=True)
    std  = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
    published = PUBLISHED[machine][mid][SNR_COLUMN[SNR]] if SNR in SNR_COLUMN else np.nan

    out = []
    for augment in AUGMENT_RUNS:
        tf.keras.backend.clear_session(); gc.collect()
        tf.keras.utils.set_random_seed(SEED)
        model, history = train_model(X_tr, y_tr, X_va, y_va_w, mean, std, augment)
        val_s  = clip_scores(model, X_va, c_va, mean, std)
        test_s = clip_scores(model, X_te, c_te, mean, std)
        m = get_metrics(val_s, y_va, test_s, y_te)
        row = {"machine": machine, "id": mid, "snr": SNR, "augment": augment, **m,
               "epochs": len(history["loss"]), "published_auc": published}
        saved = {"history": history, "y_val": y_va, "val_scores": val_s, "y_test": y_te, "test_scores": test_s, "metrics": m}
        out.append((row, saved))
        print(f"  augment={augment!s:5} | AUC {m['auc']:.3f} | bal.acc {m['bal_acc']:.3f} | recall {m['recall']:.3f} | false alarm {m['false_alarm']:.3f} | {row['epochs']} epochs")
    return out

## Step 15 - Run everything

Results are saved to `results_sup.csv` after **every** run.

In [ ]:
results, SAVED = [], {}
start = time.time()

for number, (machine, mid) in enumerate(plan, 1):
    if (time.time() - start) / 3600 > MAX_HOURS:
        print("Time limit reached. Stopping.")
        break
    print(f"[{number}/{len(plan)}] {machine} {mid}  ({(time.time() - start) / 60:.1f} min so far)")
    for row, saved in run_one(machine, mid):
        results.append(row)
        SAVED[(machine, mid, row["augment"])] = saved
    pd.DataFrame(results).to_csv(WORK_DIR + "/results_sup.csv", index=False)

res = pd.DataFrame(results)
print(f"Done in {(time.time() - start) / 60:.1f} minutes")

# Part A - Report for each machine

Each row of charts shows one machine, for the main setting (`HEADLINE_AUG`):
1. **Loss curve** - solid = train, dashed = validation (one colour per ID).
2. **Accuracy curve** - same style.
3. **ROC curve** - one line per ID.
4. **Confusion matrix** - all IDs of this machine added together, at the validation threshold.
5. **Score distribution** - healthy against faulty test clips, with the threshold.

In [ ]:
id_colors = dict(zip(sorted(set(k[1] for k in SAVED)), plt.cm.tab10.colors))

def machine_report(machine):
    runs = [k for k in SAVED if k[0] == machine and k[2] == HEADLINE_AUG]
    if not runs:
        print("No runs for", machine); return
    fig, axs = plt.subplots(1, 5, figsize=(24, 4.2))
    cm = np.zeros((2, 2), int)
    all_h, all_f, thr = [], [], []
    for k in runs:
        r = SAVED[k]; h = r["history"]; c = id_colors[k[1]]
        axs[0].plot(h["loss"], color=c, label=k[1]);     axs[0].plot(h["val_loss"], "--", color=c)
        axs[1].plot(h["accuracy"], color=c, label=k[1]); axs[1].plot(h["val_accuracy"], "--", color=c)
        fpr, tpr, _ = roc_curve(r["y_test"], r["test_scores"])
        axs[2].plot(fpr, tpr, color=c, label=f"{k[1]} (AUC {r['metrics']['auc']:.2f})")
        m = r["metrics"]; cm += np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        all_h += list(r["test_scores"][r["y_test"] == 0]); all_f += list(r["test_scores"][r["y_test"] == 1]); thr.append(m["threshold"])
    axs[0].set(title=f"{machine}: loss", xlabel="epoch", ylabel="loss"); axs[0].legend()
    axs[1].set(title=f"{machine}: accuracy", xlabel="epoch", ylabel="accuracy"); axs[1].legend()
    axs[2].plot([0, 1], [0, 1], "k:", lw=1); axs[2].set(title=f"{machine}: ROC", xlabel="false alarm rate", ylabel="faults caught"); axs[2].legend()
    ConfusionMatrixDisplay(cm, display_labels=["healthy", "faulty"]).plot(ax=axs[3], cmap="Blues", colorbar=False)
    axs[3].set_title(f"{machine}: confusion matrix (all IDs)")
    bins = np.linspace(0, 1, 31)
    axs[4].hist(all_h, bins=bins, alpha=0.6, color="tab:green", label="healthy"); axs[4].hist(all_f, bins=bins, alpha=0.6, color="tab:red", label="faulty")
    axs[4].axvline(np.mean(thr), color="k", ls="--", label="threshold"); axs[4].set(title=f"{machine}: scores", xlabel="chance of faulty"); axs[4].legend()
    show(f"machine_{machine}")
    cols = ["id", "auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa", "epochs"]
    display(res[(res.machine == machine) & (res.augment == HEADLINE_AUG)][cols].round(3).reset_index(drop=True))

## Fan

In [ ]:
machine_report("fan")

## Pump

In [ ]:
machine_report("pump")

## Slider

In [ ]:
machine_report("slider")

## Valve

In [ ]:
machine_report("valve")

# Part B - Report for all machines together

The same charts, with more detail, for **all runs**.

### B1 - Summary table (every run)

In [ ]:
cols = ["machine", "id", "augment", "auc", "pauc", "f1", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa", "epochs"]
display(res[cols].round(3).reset_index(drop=True))
print("Mean per machine and augmentation:")
display(res.groupby(["machine", "augment"])[["auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm"]].mean().round(3))
print("Overall mean per augmentation:")
display(res.groupby("augment")[["auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm"]].mean().round(3))

### B2 - Confusion matrices

Left: **all runs added together** (main setting). Right: one matrix per machine, shown as **percent of each true class** (rows add to 100%).

In [ ]:
head = res[res.augment == HEADLINE_AUG]
def matrix(df):
    return np.array([[df.tn.sum(), df.fp.sum()], [df.fn.sum(), df.tp.sum()]])

fig, axs = plt.subplots(1, 5, figsize=(22, 4.2))
ConfusionMatrixDisplay(matrix(head), display_labels=["healthy", "faulty"]).plot(ax=axs[0], cmap="Blues", colorbar=False)
axs[0].set_title("ALL machines (counts)")
for ax, machine in zip(axs[1:], MACHINES):
    sub = head[head.machine == machine]
    if len(sub) == 0:
        ax.axis("off"); continue
    cm = matrix(sub); pct = cm / cm.sum(axis=1, keepdims=True) * 100
    ConfusionMatrixDisplay(pct, display_labels=["healthy", "faulty"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format=".0f")
    ax.set_title(f"{machine} (% of true class)")
show("confusion_all")

### B3 - Confusion matrix for every run

In [ ]:
ids = sorted(head["id"].unique())
fig, axs = plt.subplots(len(MACHINES), len(ids), figsize=(4 * len(ids), 3.6 * len(MACHINES)), squeeze=False)
for i, machine in enumerate(MACHINES):
    for j, mid in enumerate(ids):
        ax = axs[i, j]
        sub = head[(head.machine == machine) & (head["id"] == mid)]
        if len(sub) == 0:
            ax.axis("off"); continue
        r = sub.iloc[0]
        ConfusionMatrixDisplay(np.array([[r.tn, r.fp], [r.fn, r.tp]]), display_labels=["healthy", "faulty"]).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(f"{machine} {mid}  (bal.acc {r.bal_acc:.2f})", fontsize=10)
show("confusion_every_run")

### B4 - Loss and accuracy curves, all runs

Colour = machine. Solid = train, dashed = validation.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for (machine, mid, aug), r in SAVED.items():
    if aug != HEADLINE_AUG: continue
    c = MACHINE_COLOR.get(machine, "gray"); h = r["history"]
    axs[0].plot(h["loss"], color=c, alpha=0.6);     axs[0].plot(h["val_loss"], "--", color=c, alpha=0.6)
    axs[1].plot(h["accuracy"], color=c, alpha=0.6); axs[1].plot(h["val_accuracy"], "--", color=c, alpha=0.6)
for ax, t in zip(axs, ["Loss", "Accuracy"]):
    ax.set(title=t + " - all runs", xlabel="epoch", ylabel=t.lower())
for machine in MACHINES:
    axs[0].plot([], [], color=MACHINE_COLOR[machine], label=machine)
axs[0].legend()
show("curves_all")

### B5 - ROC curves

Thin lines = single runs. Thick line = all clips of that machine pooled.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for machine in MACHINES:
    ys, ss = [], []
    for (mc, mid, aug), r in SAVED.items():
        if mc != machine or aug != HEADLINE_AUG: continue
        fpr, tpr, _ = roc_curve(r["y_test"], r["test_scores"])
        ax.plot(fpr, tpr, color=MACHINE_COLOR[machine], alpha=0.3, lw=1)
        ys.append(r["y_test"]); ss.append(r["test_scores"])
    if ys:
        y, s = np.concatenate(ys), np.concatenate(ss)
        fpr, tpr, _ = roc_curve(y, s)
        ax.plot(fpr, tpr, color=MACHINE_COLOR[machine], lw=2.5, label=f"{machine} pooled (AUC {roc_auc_score(y, s):.2f})")
ax.plot([0, 1], [0, 1], "k:", lw=1); ax.axvline(0.05, color="gray", ls=":", lw=1)
ax.set(xlabel="false alarm rate", ylabel="faults caught", title="ROC - all machines"); ax.legend(loc="lower right")
show("roc_all")

### B6 - Score distributions per machine

In [ ]:
fig, axs = plt.subplots(1, len(MACHINES), figsize=(5 * len(MACHINES), 3.8), sharey=False)
bins = np.linspace(0, 1, 31)
for ax, machine in zip(np.atleast_1d(axs), MACHINES):
    h, f = [], []
    for (mc, mid, aug), r in SAVED.items():
        if mc != machine or aug != HEADLINE_AUG: continue
        h += list(r["test_scores"][r["y_test"] == 0]); f += list(r["test_scores"][r["y_test"] == 1])
    ax.hist(h, bins=bins, alpha=0.6, color="tab:green", label="healthy"); ax.hist(f, bins=bins, alpha=0.6, color="tab:red", label="faulty")
    ax.set(title=machine, xlabel="chance of faulty"); ax.legend()
show("scores_per_machine")

### B7 - Heatmaps (machine x ID)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 3.8))
def heat(ax, table, title, vmin, vmax, cmap):
    im = ax.imshow(table.values, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set(xticks=range(table.shape[1]), xticklabels=table.columns, yticks=range(table.shape[0]), yticklabels=table.index, title=title)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            v = table.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if abs(v - (vmin + vmax) / 2) > (vmax - vmin) * 0.3 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
heat(axs[0], head.pivot(index="machine", columns="id", values="auc"), "AUC", 0.5, 1.0, "viridis")
heat(axs[1], head.pivot(index="machine", columns="id", values="bal_acc"), "Balanced accuracy", 0.5, 1.0, "viridis")
heat(axs[2], head.pivot(index="machine", columns="id", values="recall_at_5pct_fa"), "Faults caught at 5% false alarm", 0.0, 1.0, "viridis")
show("heatmaps")

### B8 - Does augmentation help?

Same split, same seed. Only the augmentation changes.
**Read this with care:** with few clips, differences of 0.02 to 0.03 can be luck.

In [ ]:
if set(res.augment.unique()) == {False, True}:
    pair = res.pivot_table(index=["machine", "id"], columns="augment", values=["auc", "bal_acc"])
    delta = pd.DataFrame({"auc_no_aug": pair["auc"][False], "auc_aug": pair["auc"][True],
                          "auc_change": pair["auc"][True] - pair["auc"][False],
                          "balacc_no_aug": pair["bal_acc"][False], "balacc_aug": pair["bal_acc"][True],
                          "balacc_change": pair["bal_acc"][True] - pair["bal_acc"][False]})
    display(delta.round(3))
    print("Runs where augmentation raised AUC:", int((delta.auc_change > 0).sum()), "of", len(delta))
    print("Mean AUC change:", round(delta.auc_change.mean(), 3), "| mean balanced-accuracy change:", round(delta.balacc_change.mean(), 3))
    g = res.groupby(["machine", "augment"])[["auc", "bal_acc"]].mean().unstack()
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, t in zip(axs, ["auc", "bal_acc"], ["Mean AUC", "Mean balanced accuracy"]):
        g[col].plot(kind="bar", ax=ax, rot=0); ax.set(title=t, ylim=(0.4, 1.0)); ax.legend(["no augmentation", "augmentation"])
        ax.axhline(0.5, color="gray", ls="--", lw=1)
    show("augmentation_effect")

### B9 - Counts of mistakes

Missed faults and false alarms, summed over the IDs of each machine (main setting).

In [ ]:
mist = head.groupby("machine")[["fn", "fp", "tp", "tn"]].sum()
mist["missed_faults_%"] = (100 * mist.fn / (mist.fn + mist.tp)).round(1)
mist["false_alarms_%"] = (100 * mist.fp / (mist.fp + mist.tn)).round(1)
display(mist)
ax = mist[["missed_faults_%", "false_alarms_%"]].plot(kind="bar", figsize=(7, 4), rot=0, color=["tab:red", "tab:orange"])
ax.set(ylabel="percent", title="Mistakes per machine")
show("mistakes")

### B10 - Our supervised result vs the published healthy-only result

**Not a fair match.** The published model is healthy-only. Ours saw faulty clips. This is only a reference line.

In [ ]:
g = head.groupby("machine")[["auc", "published_auc"]].mean()
ax = g.plot(kind="bar", figsize=(7, 4), rot=0)
ax.axhline(0.5, color="gray", ls="--", lw=1); ax.set(ylabel="mean AUC", ylim=(0.3, 1.02), title="Supervised (ours) vs published healthy-only")
ax.legend(["ours (supervised)", "published (healthy-only)"])
show("vs_published")

## Save files

- `results_sup.csv` = one row per run.
- `clip_scores_sup.csv` = one row per validation and test clip.
- `history_sup.csv` = loss and accuracy per epoch.
- `figures/` = every chart as a PNG.
- `export_sup.zip` = all of the above.

In [ ]:
clip_rows, hist_rows = [], []
for (machine, mid, aug), r in SAVED.items():
    thr = r["metrics"]["threshold"]
    for split, y, s in [("val", r["y_val"], r["val_scores"]), ("test", r["y_test"], r["test_scores"])]:
        clip_rows.append(pd.DataFrame({"machine": machine, "id": mid, "augment": aug, "split": split,
                                       "clip_index": np.arange(len(s)), "label": y, "score": s, "alarm": (s > thr).astype(int)}))
    h = pd.DataFrame(r["history"]); h.insert(0, "epoch", range(1, len(h) + 1))
    h.insert(0, "augment", aug); h.insert(0, "id", mid); h.insert(0, "machine", machine)
    hist_rows.append(h)

EXP = WORK_DIR + "/export_sup"
os.makedirs(EXP, exist_ok=True)
res.to_csv(EXP + "/results_sup.csv", index=False)
pd.concat(clip_rows).to_csv(EXP + "/clip_scores_sup.csv", index=False)
pd.concat(hist_rows).to_csv(EXP + "/history_sup.csv", index=False)
shutil.copytree(FIG_DIR, EXP + "/figures", dirs_exist_ok=True)
shutil.make_archive(WORK_DIR + "/export_sup", "zip", EXP)
print("Saved:", os.listdir(EXP), "-> /kaggle/working/export_sup.zip")

## Summary to paste back

Run this cell and copy its output.

In [ ]:
print("SPLIT_MODE:", SPLIT_MODE, "| LIMIT:", LIMIT, "| IDS:", IDS, "| HEADLINE_AUG:", HEADLINE_AUG)
print("=== EVERY RUN ===")
print(res[cols].round(3).to_csv(index=False))
print("=== MEAN PER MACHINE AND AUGMENTATION ===")
print(res.groupby(["machine", "augment"])[["auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa"]].mean().round(3).to_csv())
print("=== OVERALL ===")
print(res.groupby("augment")[["auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm"]].mean().round(3).to_csv())
print("=== MISTAKE COUNTS (main setting) ===")
print(mist.to_csv())